## Import Libraries

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import matplotlib.pyplot as plt
from cellpose import models, io
import numpy as np
from skimage import segmentation, filters, morphology, feature
from skimage.measure import label, regionprops
from skimage.filters import sobel, gaussian, threshold_otsu, rank
from scipy.ndimage import distance_transform_edt
import pandas as pd
import tifffile as tiff

## Paths

The path directories are adjusted to the file structure and filenames when acquiring images on the cellSens software from Evident in the "tiles" mode. Thus, images are organized in "Cycles". The original file structure was as follows:

experiment_number -> cycle_folder  -> frame_folder -> frames

In each frame folder a gray scale image of each channel and a mask from the machine learning step need to exist

In [ ]:
# sample
probe = "NR_0"
# Experiment Number
no = 5

base_path = f"YOUR_PATH/_Exp_{no}"

# find all cycle folders
cycle_folders = [f for f in os.listdir(base_path) if probe in f]
cycle_folders = sorted(cycle_folders)

print(cycle_folders)

In [ ]:
# find all frames
for cycle in cycle_folders:
    cycle_path = os.path.join(base_path, cycle)

    
    frame_folders = [f for f in os.listdir(cycle_path) if f.endswith(".frames")]
    
    for frame in frame_folders:
        frame_path = os.path.join(cycle_path, frame)

## Analysis

Define the cellpose model

In [ ]:
 model_cyto = models.Cellpose(model_type="cyto2", gpu=False)

### Define functions

Start with **debris removal**. Each label was assigned number:

- label_1 = debris
- label_2 = apoptotic cells
- label_3 = nuclei
- label_4 = cytosol
- label_5 = background

Only labels 3 and 4 are kept and used to create a new mask (living_cell_mask). The debris often "bleeds" at the edges of debris labels so the mask is dilated to fully exclude all debris signal.

In [ ]:
def remove_debris(dapi_input, lyso_input, endo_input, nr_input, debris_mask):
    # Keep living cells
    living_cell_mask = np.isin(debris_mask, [3, 4]) #creates a binary mask and sets labels 3 and 4 to 1 and the rest to 0
    living_cell_mask_dilated = morphology.binary_erosion(living_cell_mask, morphology.disk(4))  
    
    # Multiply binary mask with each channel to exclude unwanted signal
    dapi=dapi_input*living_cell_mask_dilated
    lyso=lyso_input*living_cell_mask_dilated
    endo=endo_input*living_cell_mask_dilated
    nr=nr_input*living_cell_mask_dilated
    return dapi, lyso, endo, nr

**Segment nuclei** using Cellpose. The function returns a new mask for only nuclei

In [ ]:
def segment_nuclei(dapi, model_cyto):
    masks, flows, style, diameter = model_cyto.eval([dapi.astype("float32")], diameter=70, flow_threshold=0.6, cellprob_threshold=0.0)
    nuclei= masks[0]
    return nuclei

**Segment cells** with watershed segmentation by using the nuclei mask as seeds and the lysosome and endosome channels as basis for segmentation

In [ ]:
def segment_cells(nuclei, lyso, endo):
    otsu=filters.threshold_otsu(lyso) #apply otsu threshold on lyso channel

    #define everything above threshold
    cell_region = (lyso.astype(np.float32)+endo.astype(np.float32))/2 > otsu*0.01 # creates a binary mask
    # note that the threshold is very low since the background had already been removed by applying the living_cell_mask to each channel

    # calculate the distance:
    # For each foreground pixel (=1), calculate the distance to the nearest background pixel (=0); background pixels remain 0
    distance = distance_transform_edt(cell_region)

    # segment cells
    cell_mask_dist = segmentation.watershed(-distance, markers=nuclei, mask=cell_region)
    cells=cell_mask_dist
    return cells

**Segment vesicles** <br>
To improve the segmentation, the background is removed by subtracting a strong Gaussian blur from each channel and applying a Gaussian blur on each channel itself

In [ ]:
def bg_correction(lyso, endo, nr):
    # smooth channels
    lyso_sm = gaussian(lyso.astype(np.float32), sigma=1)
    endo_sm = gaussian(endo.astype(np.float32), sigma=1)
    nr_sm = gaussian(nr.astype(np.float32), sigma=1)

    #create a background image
    bgr_lyso = gaussian(lyso_sm, sigma=7)
    bgr_endo = gaussian(endo_sm, sigma=7)
    nr_bgr = gaussian(nr_sm, sigma=7)

    #subtract background
    lyso_cor = np.clip(lyso_sm - (0.5*bgr_lyso), 0, None) # the lysosome signal was not very clean so only 50% of the background are removed to avoid removing too much signal
    endo_cor = np.clip(endo_sm - bgr_endo, 0, None)
    nr_cor = np.clip(nr_sm - nr_bgr, 0, None)
    
    return lyso_cor, endo_cor, nr_cor # return background corrected images

NR segmentation by using watershed segmentation with local maxima as seeds. The threshold is divided in a relative threshold (per cell) and absolute threhold (global). The highest of both threholds will be applied as final threshold. 

In [ ]:
def nr_segmentation(mask, nr_cor, nr):
    nr_pixel = nr_cor[mask]  # 'mask' is later defined as a mask for a single cell so nr_pixel only includes pixels from one cell

    if nr_pixel.size == 0:
        print("no pixels found")
        return

    #Define threshold
    rel_thresh_nr = np.percentile(nr_pixel, 90) #relative threshold, applied per cell
    abs_thresh_nr=80 # absolute threshold
    thresh_final_nr=max(abs_thresh_nr, rel_thresh_nr)
        
    #detect pixels above threshold
    nr_vesicle_thresh = (nr_cor > thresh_final_nr) & mask

    #Remove small objects to reduce noise
    nr_vesicle = morphology.remove_small_objects(nr_vesicle_thresh, min_size=2)

    # find local maxima
    local_max_nr = feature.peak_local_max(nr_cor, min_distance=2, labels=nr_vesicle)

    #Transform seeds
    seeds_nr = np.zeros_like(nr_vesicle, dtype=bool)
    seeds_nr[tuple(local_max_nr.T)] = True

    seeds_nr=morphology.label(seeds_nr)

    #estimate distances between vesicles in mask
    distance_nr = distance_transform_edt(nr_vesicle)

    #watershed segmentation
    nr_segmented = segmentation.watershed(-distance_nr, markers=seeds_nr, mask=nr_vesicle)
                  
    return nr_segmented

Segment Endosomes and Lysosomes using watershed segmentation with local maxima as seeds. The same thresholding method as for the NR segmentation is applied. Both absoulte threshold and relative threshold must be adjusted visually beforehand.

In [ ]:
def endo_lyso_segmentation(mask, lyso_cor, endo_cor):
    results=[]
        
    lyso_pixel = lyso_cor[mask]  # only pixels from cells
    endo_pixel = endo_cor[mask]

    if lyso_pixel.size == 0 or endo_pixel.size == 0:
        print("no pixels found in")
        return

    #Define threshold
    rel_thresh_lyso = np.percentile(lyso_pixel, 95) #relative threshold
    rel_thresh_endo = np.percentile(endo_pixel, 95)

    abs_thresh_lyso=120
    abs_thresh_endo=450

    thresh_final_lyso=max(abs_thresh_lyso, rel_thresh_lyso)
    thresh_final_endo=max(abs_thresh_endo, rel_thresh_endo)
        
    #detect only vesicles above threshold
    lyso_vesicle_thresh = (lyso_cor > thresh_final_lyso) & mask
    endo_vesicle_thresh = (endo_cor > thresh_final_endo) & mask

    #Remove small objects to reduce noise
    lyso_vesicle = morphology.remove_small_objects(lyso_vesicle_thresh, min_size=3)
    endo_vesicle = morphology.remove_small_objects(endo_vesicle_thresh, min_size=3)

    # find local maxima
    local_max_endo = feature.peak_local_max(endo_cor, min_distance=1, labels=mask)
    local_max_lyso = feature.peak_local_max(lyso_cor, min_distance=1, labels=mask)

    #Transform seeds
    seeds_endo = np.zeros_like(mask, dtype=bool)
    seeds_endo[tuple(local_max_endo.T)] = True

    seeds_lyso = np.zeros_like(mask, dtype=bool)
    seeds_lyso[tuple(local_max_lyso.T)] = True
    
    seeds_endo=morphology.label(seeds_endo)
    seeds_lyso=morphology.label(seeds_lyso)

    #estimate distances between vesicles in mask
    distance_endo = distance_transform_edt(endo_vesicle)
    distance_lyso = distance_transform_edt(lyso_vesicle)

    #watershed segmentation
    endo_vesicle_watershed = segmentation.watershed(-distance_endo, markers=seeds_endo, mask=endo_vesicle)
    lyso_vesicle_watershed = segmentation.watershed(-distance_lyso, markers=seeds_lyso, mask=lyso_vesicle)

    return endo_vesicle_watershed, lyso_vesicle_watershed

**Quantify uptake and overlap and save results** <br>

The watershed segmentation creates a mask that is not binary, meaning region properties can be extracted. The regionprops can be used to calculate size, mean intensities, etc. To calculate the sum of signal per image inside a mask, it is faster to transform it to a binary mask first.

In [ ]:
def vesicle_overlap(mask, endo_vesicle_watershed, lyso_vesicle_watershed, nr_segmented, nr):
    results= []
    #make watershed transformed array binary
    endo_vesicle_binary = endo_vesicle_watershed > 0
    lyso_vesicle_binary = lyso_vesicle_watershed > 0
    nr_vesicle_binary = nr_segmented > 0
    
    #define properties
    props_nr = regionprops(nr_segmented, nr) # Regionproperties of the Nanorod vesicles
    props_nr_in_endo=regionprops(endo_vesicle_watershed, nr) # region properties of NR signal overlapping with endosomes
    props_nr_in_lyso=regionprops(lyso_vesicle_watershed, nr) # region properties of NR signal overlapping with lysosomes

    # total nr signal for unsegmented cells
    nr_total_signal = np.sum(nr[mask])
    nr_mean_signal_per_cell = np.sum(nr[mask])/np.sum(mask)

    ######################################################
    
    #positive selection of cells with endosome or lysosome or nr
    endo_vesicle_amount = len(props_nr_in_endo)
    lyso_vesicle_amount = len(props_nr_in_lyso)
    nr_vesicle_amount = len(props_nr)
    
    has_endo=endo_vesicle_amount>0
    has_lyso=lyso_vesicle_amount>0
    has_nr=nr_vesicle_amount>0

    if has_nr:
        #vesicle size
        nr_vesicle_size = [r.area for r in props_nr]
        
        nr_vesicle_size_mean = np.mean(nr_vesicle_size)
        nr_vesicle_size_median = np.median(nr_vesicle_size)
    
        #vesicle intensity
        #sum of intensity per vesicle
        nr_total_vesicle_intensity = [v.intensity_image.sum() for v in props_nr]
        
        nr_total_vesicle_intensity_median = np.median(nr_total_vesicle_intensity)
        nr_total_vesicle_intensity_mean = np.mean(nr_total_vesicle_intensity)
        nr_total_vesicle_intensity_max = np.max(nr_total_vesicle_intensity)
    
        #mean intensity per vesicle
        nr_mean_vesicle_intensity = [m.mean_intensity for m in props_nr]
    
        nr_mean_vesicle_intensity_median = np.median(nr_mean_vesicle_intensity)
        nr_mean_vesicle_intensity_mean = np.mean(nr_mean_vesicle_intensity)
        nr_mean_vesicle_intensity_max = np.max(nr_mean_vesicle_intensity)
    
        # fraction  of pixels that were segmented
        nr_vesicle_pixels = np.sum(nr[nr_segmented > 0])  
        nr_vesicle_fraction = nr_vesicle_pixels / nr_total_signal

    else:
        nr_vesicle_size_mean = 0
        nr_vesicle_size_median = 0

        nr_total_vesicle_intensity_median = np.nan
        nr_total_vesicle_intensity_mean = np.nan
        nr_total_vesicle_intensity_max = np.nan

        nr_mean_vesicle_intensity_median = np.nan
        nr_mean_vesicle_intensity_mean = np.nan
        nr_mean_vesicle_intensity_max = np.nan

        nr_vesicle_fraction = 0

    if has_endo:
        endo_size=[e.area for e in props_nr_in_endo]
        endo_area = np.sum(endo_size)
        
        nr_in_endo_int= [e.mean_intensity for e in props_nr_in_endo]
        nr_in_endo_meanint_median = np.median(nr_in_endo_int)
        
        nr_in_endo_totalint = [v.intensity_image.sum() for v in props_nr_in_endo]
        nr_in_endo_totalint_median=np.median(nr_in_endo_totalint)
    else:
        endo_area=0 
        nr_in_endo_meanint_median= np.nan
        nr_in_endo_totalint_median = np.nan

    
    if has_lyso:
        lyso_size=[l.area for l in props_nr_in_lyso]
        lyso_area = np.sum(lyso_size)
        
        nr_in_lyso_int= [l.mean_intensity for l in props_nr_in_lyso]
        nr_in_lyso_meanint_median = np.median(nr_in_lyso_int)
        
        nr_in_lyso_totalint = [v.intensity_image.sum() for v in props_nr_in_lyso]
        nr_in_lyso_totalint_median=np.median(nr_in_lyso_totalint)
    else:
        lyso_area=0 
        nr_in_lyso_meanint_median= np.nan
        nr_in_lyso_totalint_median = np.nan

    ###OVERLAP######

    if has_endo and has_nr:
        overlap_list_endo_nr = []

        # Calculate the fraction of signal that is overlapping
        endo_nr_intersection = endo_vesicle_binary & nr_vesicle_binary # new mask which only includes intersecting vesicles
        endo_nr_frac = endo_nr_intersection.sum()/nr_vesicle_binary.sum() # Fraction of NR pixels that are colocalized with endosomes
        endo_nr_intersection_pixels = endo_nr_intersection.sum() # sum of NR pixels that are colocalized with endosomes

        nr_intensity_overlap_endo = np.sum(nr[endo_vesicle_binary & nr_vesicle_binary]) # Fraction of NR signal that is colocalized with endosomes
        overlap_endo_intensity_frac = nr_intensity_overlap_endo / np.sum(nr_total_vesicle_intensity)  # sum of NR signal that is colocalized with endosomes

        # To be able to look at the distribution of overlap and to apply an overlap threshold after the pipeline, every overlapping event needs to be saved
        for v_nr in regionprops(nr_segmented):
            mask_nr = nr_segmented == v_nr.label # every NR vesicle gets a label with its own mask
            
            # vesicle overlap
            overlapping_endo = np.unique(endo_vesicle_watershed[mask_nr])
            overlapping_endo = overlapping_endo[overlapping_endo > 0]
            
            for label_endo in overlapping_endo:
                mask_endo = endo_vesicle_watershed == label_endo # every endosome gets a label with its own mask
                overlap_pixels_endo = np.sum(mask_nr & mask_endo) 

                
                overlap_list_endo_nr.append({
                    'nr_label': int(v_nr.label),
                    'endo_label': int(label_endo),
                    'overlap_pixels': int(overlap_pixels_endo), # Overlapping Pixels
                    'overlap_frac_of_nr_in_endo': float(overlap_pixels_endo / v_nr.area),      # Fraction of NRs
                    'overlap_frac_of_endo_has_nr': float(overlap_pixels_endo / np.sum(mask_endo))  # Fraction of Endosomes
                    
                })

    elif has_endo and not has_nr:
        endo_nr_frac = 0
        overlap_list_endo_nr=[]
        endo_nr_intersection_pixels=0
        nr_intensity_overlap_endo =0
        overlap_endo_intensity_frac= 0

    elif has_nr and not has_endo:
        endo_nr_frac = np.nan
        overlap_list_endo_nr =[]
        endo_nr_intersection_pixels=np.nan
        nr_intensity_overlap_endo =np.nan
        overlap_endo_intensity_frac= np.nan

    else:
        endo_nr_frac = np.nan
        overlap_list_endo_nr=[]
        endo_nr_intersection_pixels=np.nan
        nr_intensity_overlap_endo =np.nan
        overlap_endo_intensity_frac= np.nan



##################### Now the same thing for lysosomes ########################################

    if has_lyso and has_nr:
        overlap_list_lyso_nr = []
        
        lyso_nr_intersection = lyso_vesicle_binary & nr_vesicle_binary
        lyso_nr_frac = lyso_nr_intersection.sum()/nr_vesicle_binary.sum()
        lyso_nr_intersection_pixels = lyso_nr_intersection.sum()
        
        nr_intensity_overlap_lyso = np.sum(nr[lyso_vesicle_binary & nr_vesicle_binary])
        overlap_lyso_intensity_frac = nr_intensity_overlap_lyso / np.sum(nr_total_vesicle_intensity)  

        for v_nr in regionprops(nr_segmented):
            mask_nr = nr_segmented == v_nr.label
            
            # vesicle overlap
            overlapping_lyso = np.unique(lyso_vesicle_watershed[mask_nr])
            overlapping_lyso = overlapping_lyso[overlapping_lyso > 0]
            
            for label_lyso in overlapping_lyso:
                mask_lyso = lyso_vesicle_watershed == label_lyso
                overlap_pixels_lyso = np.sum(mask_nr & mask_lyso)

                
                overlap_list_lyso_nr.append({
                    'nr_label': int(v_nr.label),
                    'lyso_label': int(label_lyso),
                    'overlap_pixels_lyso': int(overlap_pixels_lyso),
                    'overlap_frac_of_nr_in_lyso': float(overlap_pixels_lyso / v_nr.area),      # Anteil von NR
                    'overlap_frac_of_lyso_has_nr': float(overlap_pixels_lyso / np.sum(mask_lyso)) 
                    
                })        
        

    elif has_lyso and not has_nr:
        lyso_nr_frac = 0
        overlap_list_lyso_nr=[]
        lyso_nr_intersection_pixels=0
        nr_intensity_overlap_lyso=0
        overlap_lyso_intensity_frac= 0

    elif has_nr and not has_lyso:
        lyso_nr_frac = np.nan
        overlap_list_lyso_nr = []
        lyso_nr_intersection_pixels= np.nan
        nr_intensity_overlap_lyso= np.nan
        overlap_lyso_intensity_frac= np.nan

    else:
        lyso_nr_frac = np.nan
        overlap_list_lyso_nr = []
        lyso_nr_intersection_pixels = np.nan
        nr_intensity_overlap_lyso= np.nan
        overlap_lyso_intensity_frac= np.nan


    # Add all the results to a list
    results.append({
        "Cell ID": cell_id,
        "Endo total area": int(endo_area),
        "Lyso total area": int(lyso_area),
        "NR total signal": int(nr_total_signal),
        "NR in Endo mean int median nr unsegm": nr_in_endo_meanint_median,
        "NR in Lyso mean int median nr unsegm": nr_in_lyso_meanint_median,
        "NR in Endo total int median nr unsegm": nr_in_endo_totalint_median,
        "NR in Lyso total int median nr unsegm": nr_in_lyso_totalint_median,
        "NR mean signal per cell": nr_mean_signal_per_cell,
        "Endo vesicle count": int(endo_vesicle_amount),
        "Lyso vesicle count": int(lyso_vesicle_amount),
        "NR vesicle count": int(nr_vesicle_amount),
        "NR mean vesicle size": float(nr_vesicle_size_mean),
        "NR median vesicle size": float(nr_vesicle_size_median),
        "NR total vesicle intensity median": nr_total_vesicle_intensity_median,
        "NR total vesicle intensity mean": float(nr_total_vesicle_intensity_mean),
        "NR total vesicle intensity max": nr_total_vesicle_intensity_max,
        "NR mean vesicle intensity median": nr_mean_vesicle_intensity_median,
        "NR mean vesicle intensity mean": float(nr_mean_vesicle_intensity_mean),
        "NR mean vesicle intensity max": nr_mean_vesicle_intensity_max,
        "NR fraction of segmented pixels": float(nr_vesicle_fraction),
        "Endo NR intersection nr segm": endo_nr_intersection_pixels,
        "Endo NR fraction per nr segm": float(endo_nr_frac),
        "Lyso NR intersection nr segm": lyso_nr_intersection_pixels,
        "Lyso NR fraction per nr segm": float(lyso_nr_frac),
        "NR intensity overlap endo": nr_intensity_overlap_endo,
        "Overlap endo intensity frac": float(overlap_endo_intensity_frac),
        "NR intensity overlap lyso": nr_intensity_overlap_lyso,
        "Overlap lyso intensity frac": float(overlap_lyso_intensity_frac),
        "List for coloc per vesicle endo": json.dumps(overlap_list_endo_nr), # single overlapping events will be saved as a list that can be extracted later
        "List for coloc per vesicle lyso": json.dumps(overlap_list_lyso_nr)})
        
    return (results) 

## Pipeline

Now, all defined functions can run throuh a pipeline

In [ ]:
results_list = []

# 1. load images

for cycle in cycle_folders:
    print(f"Start Cycle: {cycle}")
    cycle_path = os.path.join(base_path, cycle)
    frame_folders = [f for f in os.listdir(cycle_path) if f.endswith(".frames")]
    
    total_frames = len(frame_folders)

    for i, frame in enumerate(frame_folders, start=1):
        frame_path = os.path.join(cycle_path, frame)
        dapi_input, lyso_input, endo_input, nr_input, debris_mask = None, None, None, None, None
        print(f"[{cycle}] Frame {i}/{total_frames} ({i/total_frames:.0%}) → {frame}")
    
                # --- load channels ---
        for file in os.listdir(frame_path):
            full_path = os.path.join(frame_path, file)
            
            if "C001" in file:
                dapi_input = tiff.imread(full_path)
            elif "C002" in file:
                lyso_input = tiff.imread(full_path)
            elif "C003" in file:
                endo_input = tiff.imread(full_path)
            elif "C004" in file:
                nr_input = tiff.imread(full_path)
            elif "mask" in file:
                debris_mask = tiff.imread(full_path)
        
        if any(x is None for x in (dapi_input, lyso_input, endo_input, nr_input, debris_mask)):
            print("missing channel:", full_path)
            continue

        # 2. Filter Debris
        dapi, lyso, endo, nr= remove_debris(dapi_input, lyso_input, endo_input, nr_input, debris_mask)

        # 3. segment for nuclei
        nuclei = segment_nuclei(dapi, model_cyto)
        
        # 4. segment for cytosol
        cells = segment_cells(nuclei, lyso, endo)

        # 5. correct background
        lyso_cor, endo_cor, nr_cor = bg_correction(lyso, endo, nr)

        #6. Segment vesicles by iterating over all cells
        props=regionprops(cells)
        for i, region in enumerate(props):
            
            cell_id = region.label
            mask = cells == cell_id
    
            nr_segmented = nr_segmentation(mask, nr_cor, nr)

            endo_vesicle_watershed, lyso_vesicle_watershed = endo_lyso_segmentation(mask, lyso_cor, endo_cor)
            results = vesicle_overlap(mask, endo_vesicle_watershed, lyso_vesicle_watershed, nr_segmented, nr)
                
            #7. debug check
            if len(results) == 0:
                print("No cells detected:", frame)
                continue
        
            # 8. Save data
            for r in results:
                results_list.append({
                    "sample": probe,
                    "cycle": cycle,
                    "frame": frame,
                    **r
                })
results_df = pd.DataFrame(results_list)

output_folder = f"YOUR_PATH/Excel Files Complete Analysis/Exp_{no}"
filename = f"Results_{probe}.csv"
filepath = os.path.join(output_folder, filename)
results_df.to_csv(filepath, index=False)

print("finished")